In [5]:
import os
import math
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

def lonlat_to_utm_batch(in_folder, out_folder=None, lon_col="lon", lat_col="lat"):
    """
    将文件夹中所有 CSV 的 WGS84 坐标转换为 UTM。
    每个文件生成新的 UTM_*.csv 文件，不覆盖原始文件。
    自动跳过没有 lon/lat 的文件或空值。
    """

    if out_folder is None:
        out_folder = os.path.join(in_folder, "controlpoints_utm_out")
        # 在这里修改文件夹名称

    os.makedirs(out_folder, exist_ok=True)

    csv_files = [f for f in os.listdir(in_folder) if f.lower().endswith(".csv")]
    if not csv_files:
        print("❌ 文件夹中没有找到 CSV 文件。")
        return

    for fname in csv_files:
        in_path = os.path.join(in_folder, fname)
        out_path = os.path.join(out_folder, "UTM_" + fname)

        # 读取 CSV
        df = pd.read_csv(in_path)

        if lon_col not in df.columns or lat_col not in df.columns:
            print(f"⚠️ 跳过 {fname}：缺少 '{lon_col}' 或 '{lat_col}' 列。")
            continue

        df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")
        df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")

        valid_df = df.dropna(subset=[lon_col, lat_col])
        if valid_df.empty:
            print(f"⚠️ 跳过 {fname}：lon/lat 全为空或无效。")
            continue

        # 根据中位数经纬度判断UTM带
        lon0 = float(valid_df[lon_col].median())
        lat0 = float(valid_df[lat_col].median())
        zone = int(math.floor((lon0 + 180) / 6) + 1)
        epsg = 32600 + zone if lat0 >= 0 else 32700 + zone

        # 转为GeoDataFrame并投影
        gdf = gpd.GeoDataFrame(
            valid_df,
            geometry=gpd.points_from_xy(valid_df[lon_col], valid_df[lat_col]),
            crs="EPSG:4326"
        ).to_crs(f"EPSG:{epsg}")

        valid_df["easting"] = gdf.geometry.x
        valid_df["northing"] = gdf.geometry.y
        valid_df["utm_zone"] = zone
        valid_df["epsg"] = epsg

        df.loc[valid_df.index, ["easting","northing","utm_zone","epsg"]] = \
            valid_df[["easting","northing","utm_zone","epsg"]]

        df.to_csv(out_path, index=False)
        print(f"✅ 已生成: {out_path}  (EPSG:{epsg}, Zone {zone})")

    print(f"\n🎉 全部完成！输出文件夹: {out_folder}")

In [6]:
# 输入：存放原始CSV的文件夹
in_folder = "/Users/a1-6/Self_program/underground_water/GF7_step/ICESat_csv/controlpoints_out"
lonlat_to_utm_batch(in_folder)

⚠️ 跳过 ATL08_20221212070923_12611702_007_01_rule_summary.csv：缺少 'lon' 或 'lat' 列。
⚠️ 跳过 ATL08_20211130131051_10551306_007_01_rule_summary.csv：缺少 'lon' 或 'lat' 列。
⚠️ 跳过 ATL08_20211115015401_08191302_007_01_rule_summary.csv：缺少 'lon' 或 'lat' 列。
✅ 已生成: /Users/a1-6/Self_program/underground_water/GF7_step/ICESat_csv/controlpoints_out/controlpoints_utm_out/UTM_ATL08_20240123234022_05522206_007_01_Rule8_full.csv  (EPSG:32650, Zone 50)
⚠️ 跳过 ATL08_20240123234022_05522206_007_01_rule_summary.csv：缺少 'lon' 或 'lat' 列。
✅ 已生成: /Users/a1-6/Self_program/underground_water/GF7_step/ICESat_csv/controlpoints_out/controlpoints_utm_out/UTM_ATL08_20221212070923_12611702_007_01_Rule6_full.csv  (EPSG:32650, Zone 50)
✅ 已生成: /Users/a1-6/Self_program/underground_water/GF7_step/ICESat_csv/controlpoints_out/controlpoints_utm_out/UTM_ATL08_20230129165410_06131806_007_01_Rule6_full.csv  (EPSG:32650, Zone 50)
✅ 已生成: /Users/a1-6/Self_program/underground_water/GF7_step/ICESat_csv/controlpoints_out/controlpoints_utm_out/UTM